In [1]:
import yfinance as yf
import pandas as pd

coins = ['BTC-USD', 'LINK-USD', 'STETH-USD', 'BCH-USD']
start_date = '2024-07-01'
end_date = '2025-07-01'

frames = []
for coin in coins:
    print(f"Downloading {coin}...")
    df_coin = yf.download(coin, start=start_date, end=end_date, interval='1d')
    df_coin['Coin'] = coin
    df_coin = df_coin.reset_index()
    frames.append(df_coin)

df_raw = pd.concat(frames, axis=0, ignore_index=True)
print(df_raw.head())
print(df_raw.columns)


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed

Price        Date         Close          High           Low          Open  \
Ticker                  BTC-USD       BTC-USD       BTC-USD       BTC-USD   
0      2024-07-01  62851.980469  63777.226562  62495.511719  62673.605469   
1      2024-07-02  62029.015625  63203.359375  61752.746094  62844.410156   
2      2024-07-03  60173.921875  62187.703125  59419.386719  62034.332031   
3      2024-07-04  56977.703125  60399.675781  56777.804688  60147.136719   
4      2024-07-05  56662.375000  57497.152344  53717.375000  57022.808594   

Price         Volume     Coin    Close     High      Low  ...     Close  \
Ticker       BTC-USD          LINK-USD LINK-USD LINK-USD  ... STETH-USD   
0       2.546838e+10  BTC-USD      NaN      NaN      NaN  ...       NaN   
1       2.015162e+10  BTC-USD      NaN      NaN      NaN  ...       NaN   
2       2.975670e+10  BTC-USD      NaN      NaN      NaN  ...       NaN   
3       4.114961e+10  BTC-USD      NaN      NaN      NaN  ...       NaN   
4       5.

In [3]:
# Only keep necessary columns
needed_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Coin']
df_keep = df_raw[needed_cols]
print(df_keep.head())
print(df_keep.columns)


Price        Date          Open                                     High  \
Ticker                  BTC-USD LINK-USD STETH-USD BCH-USD       BTC-USD   
0      2024-07-01  62673.605469      NaN       NaN     NaN  63777.226562   
1      2024-07-02  62844.410156      NaN       NaN     NaN  63203.359375   
2      2024-07-03  62034.332031      NaN       NaN     NaN  62187.703125   
3      2024-07-04  60147.136719      NaN       NaN     NaN  60399.675781   
4      2024-07-05  57022.808594      NaN       NaN     NaN  57497.152344   

Price                                       Low  ...                 Close  \
Ticker LINK-USD STETH-USD BCH-USD       BTC-USD  ... BCH-USD       BTC-USD   
0           NaN       NaN     NaN  62495.511719  ...     NaN  62851.980469   
1           NaN       NaN     NaN  61752.746094  ...     NaN  62029.015625   
2           NaN       NaN     NaN  59419.386719  ...     NaN  60173.921875   
3           NaN       NaN     NaN  56777.804688  ...     NaN  56977.703125   

In [3]:
# If your columns are a MultiIndex, flatten them to normal string names
if isinstance(df_keep.columns, pd.MultiIndex):
    df_keep.columns = ['_'.join(filter(None, map(str, col))).strip('_') for col in df_keep.columns.values]
print(df_keep.columns)


Index(['Date', 'Open_BTC-USD', 'Open_LINK-USD', 'Open_STETH-USD',
       'Open_BCH-USD', 'High_BTC-USD', 'High_LINK-USD', 'High_STETH-USD',
       'High_BCH-USD', 'Low_BTC-USD', 'Low_LINK-USD', 'Low_STETH-USD',
       'Low_BCH-USD', 'Close_BTC-USD', 'Close_LINK-USD', 'Close_STETH-USD',
       'Close_BCH-USD', 'Volume_BTC-USD', 'Volume_LINK-USD',
       'Volume_STETH-USD', 'Volume_BCH-USD', 'Coin'],
      dtype='object')


In [5]:
# Identify coins
coins = ['BTC-USD', 'LINK-USD', 'STETH-USD', 'BCH-USD']

# Parse columns for each coin
records = []
for idx, row in df_keep.iterrows():
    date = row['Date']
    for coin in coins:
        open_col = f'Open_{coin}'
        high_col = f'High_{coin}'
        low_col = f'Low_{coin}'
        close_col = f'Close_{coin}'
        volume_col = f'Volume_{coin}'
        if open_col in df_keep.columns and pd.notnull(row[open_col]):
            records.append({
                'Date': date,
                'Coin': coin,
                'Open': row[open_col],
                'High': row[high_col],
                'Low': row[low_col],
                'Close': row[close_col],
                'Volume': row[volume_col],
            })

df_long = pd.DataFrame(records)
print(df_long.head())
print(df_long['Coin'].value_counts())


Empty DataFrame
Columns: []
Index: []


KeyError: 'Coin'

In [5]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']

print("Missing value counts per feature:")
print(df_long[features].isnull().sum())

# Only clean if missing values exist
if df_long[features].isnull().sum().sum() > 0:
    df_long = df_long.sort_values(['Coin', 'Date'])
    df_long[features] = df_long.groupby('Coin')[features].transform(lambda x: x.ffill().bfill())
    # Final median fill as backup
    for feat in features:
        df_long[feat] = df_long[feat].fillna(df_long[feat].median())
    print("Missing values after fill:")
    print(df_long[features].isnull().sum())
else:
    print("No missing values found!")


Missing value counts per feature:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64
No missing values found!


In [6]:
import numpy as np

def cap_outliers(group):
    for feat in features:
        Q1 = group[feat].quantile(0.25)
        Q3 = group[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        group[feat] = np.clip(group[feat], low, high)
    return group

# Outlier counts
print("\nOutlier counts before capping:")
for coin, group in df_long.groupby('Coin'):
    outliers = {}
    for feat in features:
        Q1 = group[feat].quantile(0.25)
        Q3 = group[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        outliers[feat] = ((group[feat] < low) | (group[feat] > high)).sum()
    print(f"{coin}: {outliers}")

# Cap outliers (only if there are any)
df_long = df_long.groupby('Coin', group_keys=False).apply(cap_outliers)

print("\nPreview after outlier capping:")
print(df_long.head())



Outlier counts before capping:
BCH-USD: {'Open': 6, 'High': 5, 'Low': 3, 'Close': 6, 'Volume': 30}
BTC-USD: {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 22}
LINK-USD: {'Open': 11, 'High': 17, 'Low': 19, 'Close': 11, 'Volume': 26}
STETH-USD: {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 16}

Preview after outlier capping:
        Date     Coin          Open          High           Low         Close  \
0 2024-07-01  BTC-USD  62673.605469  63777.226562  62495.511719  62851.980469   
1 2024-07-02  BTC-USD  62844.410156  63203.359375  61752.746094  62029.015625   
2 2024-07-03  BTC-USD  62034.332031  62187.703125  59419.386719  60173.921875   
3 2024-07-04  BTC-USD  60147.136719  60399.675781  56777.804688  56977.703125   
4 2024-07-05  BTC-USD  57022.808594  57497.152344  53717.375000  56662.375000   

         Volume  
0  2.546838e+10  
1  2.015162e+10  
2  2.975670e+10  
3  4.114961e+10  
4  5.541754e+10  


/var/folders/rs/5fvy4hss7cvgk5msj3vhn6t80000gn/T/ipykernel_83276/2935926726.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_long = df_long.groupby('Coin', group_keys=False).apply(cap_outliers)


In [ ]:
# df_long.to_csv('../data/cleaned/cleaned_enriched_dataset.csv', index=False)
print("✅ Final cleaned enriched dataset saved at '../data/preprocessed/cleaned_enriched_dataset1.csv'")
